In [2]:
# pip install selenium pandas

<h1> Setting Up

In [3]:
import pandas as pd

import time
from datetime import datetime, timedelta

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException

<h1> Scrape Booking.com's Data

<h2> Get Scraping's Time and Link

In [4]:
checkin_date = datetime.now() + timedelta(days=14)
checkout_date = checkin_date + timedelta(days=3)

checkin_str = checkin_date.strftime('%Y-%m-%d')
checkout_str = checkout_date.strftime('%Y-%m-%d')

# ur = "https://www.booking.com/searchresults.html?ss=London&group_adults=2&ltfd=1%3A5%3A11-2026%3A1%3A&no_rooms=1&group_children=0&selected_currency=USD"
url = (
    f"https://www.booking.com/searchresults.html?"
    f"ss=London&"
    f"group_adults=2&"
    f"no_rooms=1&"
    f"group_children=0&"
    f"checkin={checkin_str}&"
    f"checkout={checkout_str}&"
    f"selected_currency=USD"
)

print(f"Scraping for dates: {checkin_str} to {checkout_str}")

Scraping for dates: 2026-10-12 to 2026-10-15


<h2> Get Raw Data

In [5]:
# --- Setting Up Chrome Driver ---
driver = webdriver.Chrome()
driver.fullscreen_window()
driver.get(url)

time.sleep(10)

# Close Sign-In Button
try:
    close_button = WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[aria-label="Dismiss sign-in info."]'))
    )
    close_button.click()
except TimeoutException:
    pass

# Load More Data
LOAD_MORE_XPATH = "//button[contains(., 'Load more results')]"
max_loads = 20
max_scroll_steps = 30

for load_num in range(max_loads):
    load_more_btn = None

    for _ in range(max_scroll_steps):
        found = driver.find_elements(By.XPATH, LOAD_MORE_XPATH)  # returns [] instead of raising
        if found:
            load_more_btn = found[0]
            break
        driver.execute_script("window.scrollBy(0, 1000);")
        time.sleep(1)

    if load_more_btn is None:
        print("No more 'Load more results' button, stopping.")
        break

    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", load_more_btn)
    time.sleep(0.5)
    driver.execute_script("arguments[0].click();", load_more_btn)
    # print(f"Clicked load more ({load_num + 1}/{max_loads})")

    time.sleep(3)
    
time.sleep(2)

# Get Raw Datas
hotel_cards = driver.find_elements(By.CSS_SELECTOR, '[data-testid="property-card"]')
print(f'Found total {len(hotel_cards)} property.')

Found total 345 property.


<h2> Parse Scraped Data

In [6]:
def get_data(card, css, attr="innerText"):
    try:
        return card.find_element(By.CSS_SELECTOR, css).get_attribute(attr)
    except NoSuchElementException:
        return None

data = []

for card in hotel_cards:
    
    # Parse ratings
    raw = get_data(card, '[data-testid="review-score"]') or ""
    parts = [p.strip() for p in raw.split('\n') if p.strip()]
    parts = ([None] * 3 + parts)[-3:]
    score, rankings, reviews = parts[-3:]
    
    data.append({
        'URL': get_data(card, '[data-testid="title-link"]', "href"),
        'Name': get_data(card, '[data-testid="title"]'),
        'Location': get_data(card, '[data-testid="address-link"]'),
        'Price': get_data(card, '[data-testid="price-and-discounted-price"]'),
        'Rating_Score': score,
        'Rating': rankings,
        'Reviews': (reviews or "").split(' ')[0],
        'Scraped_At': datetime.now().isoformat(),
        'Search_URL': driver.current_url,
    })

<h2> Finishing

In [7]:
driver.quit()

# Convert to DataFrame
df = pd.DataFrame(data)
display(df.head())

,URL,Name,Location,Price,Rating_Score,Rating,Reviews,Scraped_At,Search_URL
0,https://www.booking.com/hotel/gb/lancaster-gat...,Lancaster Gate Hotel,"Westminster Borough, London",US$865,7.2,Good,"9,217",2026-09-28T17:54:09.027554,https://www.booking.com/searchresults.html?ss=...
1,https://www.booking.com/hotel/gb/hub-london-cl...,hub London Clerkenwell,"Islington, London",US$832,8.2,Very Good,56,2026-09-28T17:54:09.083155,https://www.booking.com/searchresults.html?ss=...
2,https://www.booking.com/hotel/gb/ibis-london-c...,ibis London City - Shoreditch,"Tower Hamlets, London","US$1,450",8.1,Very Good,"6,552",2026-09-28T17:54:09.142151,https://www.booking.com/searchresults.html?ss=...
3,https://www.booking.com/hotel/gb/riu-plaza-lon...,Riu Plaza London Victoria,"Westminster Borough, London","US$1,494",8.6,Excellent,"15,376",2026-09-28T17:54:09.202854,https://www.booking.com/searchresults.html?ss=...
4,https://www.booking.com/hotel/gb/hub-london-pa...,hub London - Paddington,"Westminster Borough, London",US$836,7.7,Good,95,2026-09-28T17:54:09.271125,https://www.booking.com/searchresults.html?ss=...


<h1> Export to CSV

In [8]:
# Export to CSV
df.to_csv('scraped_data.csv', index=False)